# EDA — Petals to the Metal (FlowerClf)

104-class flower classification, metric **macro F1**. Run this **after** `src/convert.py`
has produced `data/processed/metadata_224.parquet` and the JPEGs (on the GPU box).

Goal: understand class imbalance (drives macro-F1 strategy), train-vs-val distribution,
and image sanity. Findings → `reports/EDA_FINDINGS.md`.

In [ ]:
import sys; sys.path.insert(0, '..')
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from src import data

RES = 224
FIG = data.ROOT / 'reports' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
df = data.load_metadata(RES)
print(df.shape); print(df['split'].value_counts().to_dict())
df.head()

## Class distribution & imbalance (train+val)

In [ ]:
lab = df[df['split'].isin(['train','val'])].copy()
counts = lab['label'].astype(int).value_counts().sort_index()
counts.index = [data.CLASSES[i] for i in counts.index]
print('classes present:', counts.shape[0], '/ 104')
print('min/median/max per class:', counts.min(), int(counts.median()), counts.max())
print('imbalance ratio (max/min):', round(counts.max()/counts.min(), 2))

fig, ax = plt.subplots(figsize=(12,4))
counts.sort_values().plot(kind='bar', ax=ax, width=1.0)
ax.set_xticks([]); ax.set_ylabel('images'); ax.set_title('Per-class counts (sorted)')
fig.tight_layout(); fig.savefig(FIG/'class_distribution.png', dpi=120); plt.show()
print('\n10 rarest classes:'); print(counts.sort_values().head(10))

## Train vs val distribution

In [ ]:
piv = (lab.assign(n=1).pivot_table(index='label', columns='split', values='n', aggfunc='sum')
          .fillna(0).astype(int))
piv['val_frac'] = piv['val'] / (piv['train'] + piv['val'])
print('val fraction per class — mean %.3f, min %.3f, max %.3f' %
      (piv['val_frac'].mean(), piv['val_frac'].min(), piv['val_frac'].max()))
piv.describe()

## Sample images & size sanity

In [ ]:
sample = lab.groupby('label', group_keys=False).head(1).head(12)
fig, axes = plt.subplots(2, 6, figsize=(14,5))
for ax, (_, r) in zip(axes.ravel(), sample.iterrows()):
    im = Image.open(data.ROOT / r['path']).convert('RGB')
    ax.imshow(im); ax.set_title(r['class_name'], fontsize=8); ax.axis('off')
fig.tight_layout(); fig.savefig(FIG/'sample_grid.png', dpi=120); plt.show()

sizes = [Image.open(data.ROOT / p).size for p in lab['path'].head(200)]
print('unique sizes in first 200:', set(sizes))

## Findings

Fill `reports/EDA_FINDINGS.md` from the numbers above: # classes present, imbalance ratio,
rarest classes, val fraction consistency, image size. These drive whether we need the
inverse-frequency sampler / class-balanced loss for macro F1.